In [2]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, AdditiveAttention, Concatenate
)
from tensorflow.keras.models import Model

# 1. Small English-Spanish dataset
english_sentences = [
    "hello", "how are you", "good morning",
    "thank you", "i am happy", "i love you",
    "good night", "see you", "i am fine",
    "what is your name", "i am learning", "goodbye"
]

spanish_sentences = [
    "hola", "como estas", "buenos dias",
    "gracias", "estoy feliz", "te amo",
    "buenas noches", "hasta luego", "estoy bien",
    "como te llamas", "estoy aprendiendo", "adios"
]

# Add start and end tokens to target sentences
spanish_sentences = [
    "<start> " + s + " <end>" for s in spanish_sentences
]

# 2. Tokenize and pad the English sentences
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(english_sentences)
eng_sequences = eng_tokenizer.texts_to_sequences(english_sentences)

max_eng_len = max(len(s) for s in eng_sequences)
encoder_input = pad_sequences(
    eng_sequences, maxlen=max_eng_len, padding="post"
)

# 3. Tokenize and pad the Spanish sentences
spa_tokenizer = Tokenizer(filters="")
spa_tokenizer.fit_on_texts(spanish_sentences)
spa_sequences = spa_tokenizer.texts_to_sequences(spanish_sentences)

max_spa_len = max(len(s) for s in spa_sequences)
spa_padded = pad_sequences(
    spa_sequences, maxlen=max_spa_len, padding="post"
)

# Teacher forcing: decoder input and expected output
decoder_input = spa_padded[:, :-1]
decoder_target = spa_padded[:, 1:]

eng_vocab = len(eng_tokenizer.word_index) + 1
spa_vocab = len(spa_tokenizer.word_index) + 1
decoder_len = max_spa_len - 1

print("English vocabulary size:", eng_vocab)
print("Spanish vocabulary size:", spa_vocab)
print("Number of training sentences:", len(english_sentences))

# 4. Encoder
encoder_inputs = Input(shape=(max_eng_len,), name="english_input")
encoder_embedding = Embedding(eng_vocab, 64, mask_zero=True)(
    encoder_inputs
)
encoder_outputs, state_h, state_c = LSTM(
    64, return_sequences=True, return_state=True
)(encoder_embedding)

# 5. Decoder
decoder_inputs = Input(shape=(decoder_len,), name="spanish_input")
decoder_embedding = Embedding(spa_vocab, 64, mask_zero=True)(
    decoder_inputs
)
decoder_outputs = LSTM(64, return_sequences=True)(
    decoder_embedding, initial_state=[state_h, state_c]
)

# 6. Attention mechanism
attention_layer = AdditiveAttention()
context = attention_layer([decoder_outputs, encoder_outputs])

combined = Concatenate(axis=-1)([decoder_outputs, context])
outputs = Dense(spa_vocab, activation="softmax")(combined)

# 7. Build and train the model
model = Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    [encoder_input, decoder_input],
    decoder_target[..., np.newaxis],
    epochs=300,
    batch_size=4,
    verbose=0
)

print("Model training completed!")

# 8. Translate English into Spanish using greedy decoding
def translate(sentence):
    sequence = eng_tokenizer.texts_to_sequences([sentence])
    sequence = pad_sequences(
        sequence, maxlen=max_eng_len, padding="post"
    )

    start_id = spa_tokenizer.word_index["<start>"]
    end_id = spa_tokenizer.word_index["<end>"]

    generated = [start_id]

    for step in range(decoder_len):
        decoder_sequence = pad_sequences(
            [generated], maxlen=decoder_len, padding="post"
        )

        predictions = model.predict(
            [sequence, decoder_sequence], verbose=0
        )

        next_id = int(np.argmax(predictions[0, step, :]))

        if next_id == 0 or next_id == end_id:
            break

        generated.append(next_id)

    words = [
        spa_tokenizer.index_word.get(i, "")
        for i in generated[1:]
    ]

    return " ".join(words)

# 9. Test the model
test_sentences = [
    "hello",
    "how are you",
    "thank you",
    "good night"
]

print("\nEnglish-to-Spanish Translation:")
for sentence in test_sentences:
    print("English:", sentence)
    print("Spanish:", translate(sentence))
    print()


English vocabulary size: 21
Spanish vocabulary size: 21
Number of training sentences: 12
Model training completed!

English-to-Spanish Translation:
English: hello
Spanish: hola

English: how are you
Spanish: como estas

English: thank you
Spanish: gracias

English: good night
Spanish: buenas noches

